In [7]:
import tensorflow as tf
import category_encoders as ce
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate
from tensorflow.keras.optimizers.schedules import PolynomialDecay
import copy
from numpy import unique
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Embedding
from keras.layers import concatenate
from sklearn.metrics import mean_squared_error
from keras.utils import plot_model
import math
from tensorflow.keras.layers import Reshape
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from transformers import XLMRobertaTokenizer, TFAutoModel
from transformers import TFAutoModelForSequenceClassification
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import LSTM
import matplotlib.pyplot as plt
#from keras.utils import plot_model
import os
from sklearn.preprocessing import RobustScaler
from collections import Counter
import tensorflow as tf
from tensorflow.keras import optimizers
from tensorflow.keras import metrics
from tensorflow.keras import losses
from transformers import pipeline
from keras.layers import Softmax
from tensorflow.keras.utils import plot_model
from tensorflow.keras import regularizers
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import pickle

In [ ]:
pip install transformers tensorflow sentencepiece torch category_encoders

In [8]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [10]:
def filter_outliers(df, upper, lower, target_col = "AWARD_VALUE_EURO_FIN_1"):
    """This function only works on numerical columns"""
    data_array = np.sort(df[target_col].to_numpy())
    cut_off_low_indice = math.floor(lower * len(data_array))
    cut_off_high_indice = math.floor(upper * len(data_array)) - 1
    low_amount = data_array[cut_off_low_indice]
    high_amount = data_array[cut_off_high_indice]

    outlier_indices = []

    for i in range(0, len(df)):
        if df[target_col].iloc[i] > high_amount:
            outlier_indices.append(i)
        elif df[target_col].iloc[i] < low_amount:
            outlier_indices.append(i)
        else:
            continue

    print("{} rows were dropped because of outliers, with high amount = {}, and low amount = {}".format(len(outlier_indices), high_amount, low_amount))
    df = df.drop(labels = outlier_indices, axis = 0)
    return df

In [11]:
def train_validate_test_split(df, train, validate):
    target_col = "AWARD_VALUE_EURO_FIN_1"
    text_col = "short description"
    #df = df.sort_values(by = ["date_conclusion_contract"], axis = 0, ascending = True)
    train_indice = int(train * len(df))
    validate_indice = int(validate * len(df)) + train_indice

    train_set = df[:train_indice]
    val_set = df[train_indice:validate_indice]
    test_set = df[validate_indice:]

    X_train = train_set.drop(columns = [target_col, text_col]).values
    X_train_text = train_set[text_col].tolist()
    y_train = train_set[target_col].values

    X_val = val_set.drop(columns = [target_col, text_col]).values
    X_val_text = val_set[text_col].tolist()
    y_val = val_set[target_col].values

    X_test = test_set.drop(columns = [target_col, text_col]).values
    X_test_text = test_set[text_col].tolist()
    y_test = test_set[target_col].values

    return X_train, X_train_text, y_train, X_val, X_val_text, y_val, X_test, X_test_text, y_test

In [12]:
def scale_data(df):
    numerical_cols = ["LOTS_NUMBER", "CRIT_PRICE_WEIGHT", "NUMBER_OFFERS", "NUMBER_TENDERS_SME"]
    scaler = RobustScaler()

    numeric_data = df[numerical_cols].values.reshape((len(df), len(numerical_cols)))
    scaled_num_data = scaler.fit_transform(numeric_data)
    df_numeric = pd.DataFrame(data=scaled_num_data,columns = numerical_cols)
    df[numerical_cols] = df_numeric

    return df

In [13]:
def encode_data(df, target_column="award_contract/val_total"):

    base_n_encoder_cols = ["ISO_COUNTRY_CODE", "MAIN_ACTIVITY", "CPV"]
    one_hot_cols = ["CAE_TYPE", "B_ON_BEHALF", "B_AWARDED_BY_CENTRAL_BODY", "TYPE_OF_CONTRACT",
                    "B_FRA_AGREEMENT", "B_EU_FUNDS", "TOP_TYPE", "CRIT_CODE", "lots"]

    encoder = ce.BaseNEncoder(cols=base_n_encoder_cols, return_df=True, base=2)
    df = encoder.fit_transform(df)
    df = pd.get_dummies(df, columns=(one_hot_cols), drop_first=True, dtype=int)

    return df

In [14]:
#load the data
df = pd.read_pickle("/content/drive/MyDrive/Thesis/df_merged_dataset_clean").reset_index()
df["short description"] = df["short description"].astype(str)

crit_price_weight_empty = list(df.loc[pd.isna(df["CRIT_PRICE_WEIGHT"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)

crit_price_weight_empty = list(df.loc[pd.isna(df["NUMBER_OFFERS"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)

crit_price_weight_empty = list(df.loc[pd.isna(df["NUMBER_TENDERS_SME"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)

In [ ]:
pd.isna(df).sum()

In [16]:
df = scale_data(df)
#ENCODE DATA
df = encode_data(df)

In [17]:
drop_cols = ["pk", "ID_NOTICE_CAN"]
df = df.drop(columns = drop_cols)

In [ ]:
df = filter_outliers(df, upper=0.90, lower=0.20)

In [ ]:
df.head()

In [20]:
X_train, X_train_text, y_train, X_val, X_val_text, y_val, X_test, X_test_text, y_test = train_validate_test_split(df, 0.6, 0.2)

In [21]:
y_train = y_train.astype("float32")
y_val = y_val.astype("float32")
y_test = y_test.astype("float32")

In [ ]:
type(y_train[0])

In [ ]:
print(len(X_train), len(X_train_text), len(y_train), "\n",
      len(X_val), len(X_val_text), len(y_val), '\n',
      len(X_test), len(X_test_text), len(y_test))

In [ ]:
y_train.shape

In [ ]:
transformer_model_name = "jplu/tf-xlm-roberta-base"
tokenizer = XLMRobertaTokenizer.from_pretrained(transformer_model_name)
max_length = 128

In [ ]:
#Tokenize training set
train_input_ids = tokenizer(X_train_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["input_ids"]
train_attention_mask = tokenizer(X_train_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["attention_mask"]

#tokenize val set
val_input_ids = tokenizer(X_val_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["input_ids"]
val_attention_mask = tokenizer(X_val_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["attention_mask"]

#tokenize test set
test_input_ids = tokenizer(X_test_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["input_ids"]
test_attention_mask = tokenizer(X_test_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["attention_mask"]

In [ ]:
#with open("/content/drive/MyDrive/Thesis/train_input_ids", "wb") as f:
#  pickle.dump(train_input_ids, f)

#with open("/content/drive/MyDrive/Thesis/train_attention_mask", "wb") as f:
#  pickle.dump(train_attention_mask, f)

#with open("/content/drive/MyDrive/Thesis/val_input_ids", "wb") as f:
#  pickle.dump(val_input_ids, f)

#with open("/content/drive/MyDrive/Thesis/val_attention_mask", "wb") as f:
#  pickle.dump(val_attention_mask, f)

#with open("/content/drive/MyDrive/Thesis/test_input_ids", "wb") as f:
#  pickle.dump(test_input_ids, f)

#with open("/content/drive/MyDrive/Thesis/test_attention_mask", "wb") as f:
#  pickle.dump(test_attention_mask, f)

In [26]:
with open("/content/drive/MyDrive/Thesis/train_input_ids", "rb") as f:
  train_input_ids = pickle.load(f)

with open("/content/drive/MyDrive/Thesis/train_attention_mask", "rb") as f:
  train_attention_mask = pickle.load(f)

with open("/content/drive/MyDrive/Thesis/val_input_ids", "rb") as f:
  val_input_ids = pickle.load(f)

with open("/content/drive/MyDrive/Thesis/val_attention_mask", "rb") as f:
  val_attention_mask = pickle.load(f)

with open("/content/drive/MyDrive/Thesis/test_input_ids", "rb") as f:
  test_input_ids = pickle.load(f)

with open("/content/drive/MyDrive/Thesis/test_attention_mask", "rb") as f:
  test_attention_mask = pickle.load(f)

In [ ]:
transformer_model_name = "jplu/tf-xlm-roberta-base"
language_model = TFAutoModelForSequenceClassification.from_pretrained(transformer_model_name, num_labels=1)
language_model.layers[0].trainable = False

In [ ]:
language_model.summary()

-----RUN THE HUGGINGFACE MODEL WITH ONLY MULTIPLE HEADS BASED ON THE ENCODING OF THE TARGET VARIABLE-----

In [ ]:
import math
#define path for saving model
checkpoint_path = "/content/drive/MyDrive/Thesis/checkpoint.ckpt"
checkpoint_dir = os.path.dirname(checkpoint_path)

#specify model parameters
batch_size = 32
num_epochs = 5

n_batches = len(X_train) / batch_size
n_batches = math.ceil(n_batches)

# Create a callback that saves the model's weights
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,save_weights_only=True,verbose=1, save_freq=n_batches)

#train the model directly on the classification task
transformer_model_name = "jplu/tf-xlm-roberta-base"
language_model = TFAutoModelForSequenceClassification.from_pretrained(transformer_model_name, num_labels=1)
language_model.layers[0].trainable = False

num_train_steps = len(y_train) / batch_size * num_epochs
lr_scheduler = PolynomialDecay(
    initial_learning_rate=0.0001, end_learning_rate=0.000001, decay_steps=num_train_steps
)

optimizer = tf.keras.optimizers.Adam(lr=lr_scheduler)
metrics = ["mae", "mse", "R2Score"]

#compile the model
language_model.compile(optimizer="Adam", metrics = metrics, loss=tf.keras.losses.MeanSquaredError())

#fit the model
history_transformer = language_model.fit(x = [train_input_ids, train_attention_mask], y=y_train,
                                        validation_data=([val_input_ids, val_attention_mask], y_val),
                                        epochs=num_epochs,
                                        batch_size = batch_size,
                                       )

In [ ]:
language_model.save_weights('/content/drive/MyDrive/Thesis/language_model.keras')

In [ ]:
with open("/content/drive/MyDrive/Thesis/history_language_model.pickle", "wb") as f:
  pickle.dump(history_transformer, f)

-----RUN THE COMBINED MODEL-----

In [ ]:
#DEFINE VARIABLES
input_dim_num_cat=X_train.shape[1]
max_length = 128

import math
#define path for saving model
checkpoint_path = "/content/drive/MyDrive/Thesis/checkpoint.ckpt"
checkpoint_dir = os.path.dirname(checkpoint_path)

#specify model parameters
batch_size = 32
num_epochs = 20

n_batches = len(X_train) / batch_size
n_batches = math.ceil(n_batches)

# Create a callback that saves the model's weights
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,save_weights_only=True,verbose=1, save_freq=n_batches)

#MAKE THE FINAL MODEL
input_num_cat = Input(shape=input_dim_num_cat)
x = Dense(32, activation = "relu")(input_num_cat)
x = Dropout(rate=0.2)(x)
x = Dense(8, activation = "relu")(x)
x = Dropout(rate=0.2)(x)
x = Dense(32, activation = "relu")(x)
x = Dropout(rate=0.2)(x)
num_cat_output = Dense(1, activation = "relu")(x)

#INITIALIZE THE TRANSFORMER MODEL
transformer_model_name = "jplu/tf-xlm-roberta-base"
transformer = TFAutoModelForSequenceClassification.from_pretrained(transformer_model_name, num_labels=1)
transformer.layers[0].trainable = False

#DEFINE INPUTS AND FEED TO THE TRANSFORMER MODEL
input_ids = tf.keras.Input(shape=(max_length,), dtype='int32')
attention_mask = tf.keras.Input(shape=(max_length,), dtype='int32')
transformer_output = transformer(input_ids, attention_mask).logits

#COMBINE THE MODEL OUTPUTS
concatentate = concatenate([num_cat_output, transformer_output])
final_layer = Dense(1, activation = "linear")(concatentate)

#create final model
final_model = Model(inputs=[{"input_ids": input_ids, "attention_mask": attention_mask}, input_num_cat],
                    outputs=[final_layer],
                    name="merged")

#DEFINE LEARNING RATE DECAY FOR OPTIMIZATION
num_train_steps = len(y_train) / batch_size * num_epochs
lr_scheduler = PolynomialDecay(initial_learning_rate=0.0001,
                               end_learning_rate=0.000001,
                               decay_steps=num_train_steps)

#define custom loss function for combining classification with regression
loss_function = keras.losses.MeanSquaredError(reduction="sum_over_batch_size", name="mean_squared_error")
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_scheduler)
metrics = ["mse", "mae", "R2Score"]

final_model.compile(loss=loss_function,
                   optimizer=optimizer,
                    metrics=metrics)

history_final_model = final_model.fit(x=[[train_input_ids, train_attention_mask], X_train], y=y_train,
                validation_data=([[val_input_ids, val_attention_mask], X_val], y_val),
                epochs=num_epochs,
                batch_size = batch_size,
                callbacks=[cp_callback])

In [23]:
with open("/content/drive/MyDrive/Thesis/history_combined_model", "wb") as f:
  pickle.dump(history_final_model.history, f)

In [21]:
final_model.save_weights('/content/drive/MyDrive/Thesis/combined_model_weights.keras')

In [25]:
#load model weights
final_model.load_weights('/content/drive/MyDrive/Thesis/combined_model_weights.keras')

In [ ]:
#use the weights to make predictions
y_pred = final_model.predict([[test_input_ids, test_attention_mask], X_test])
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

metric = tf.keras.metrics.R2Score()
metric.update_state(y_test.reshape(-1,1), y_pred)
r2_result = metric.result().numpy()


In [ ]:
print(mae, mse, r2_result)

previous results: 124896.4 47251132000.0 -0.48775887

In [ ]:
#files.download("/content/drive/MyDrive/Thesis/history_final_model")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
results_dict = history_final_model.history

In [ ]:
with open("/content/drive/MyDrive/Thesis/results_dict.pickle", "wb") as f:
  pickle.dump(results_dict, f)

In [ ]:
final_model.summary()

In [ ]:
plot_metrics(ledger=history_final_model)

In [ ]:
from tensorflow.keras.utils import plot_model
keras.utils.plot_model(final_model, show_shapes=True)

In [ ]:
amount_of_targets = 6
max_length = 128

#define the numeric and categorical model
input_dim_num_cat = 55 #len(X_train[0])
input_num_cat = Input(shape=input_dim_num_cat)
num_cat_layer = Dense(68, activation = "relu", kernel_regularizer=regularizers.L1L2(l1=0.001))(input_num_cat)
dropout_layer = Dropout(rate=0.3)(num_cat_layer)
num_cat_layer = Dense(32, activation="relu", kernel_regularizer=regularizers.L1L2(l1=0.001))(dropout_layer)
num_cat_layer_final = Dense(8, activation="relu")(num_cat_layer)

input_ids = tf.keras.Input(shape=(max_length,), dtype='int32')
attention_mask = tf.keras.Input(shape=(max_length, ), dtype='int32')
encoding_transformer = transformer(input_ids, attention_mask)
logits = encoding_transformer[0]
probabilities = Dense(amount_of_targets, activation="softmax")(logits)

#concat layers
combined_input = concatenate([probabilities, num_cat_layer_final], name="concatenated_layer")
final_layer = Dense(1, activation="linear", name="output_layer")(combined_input)

#create final model
final_model = Model(inputs=[{"input_ids": input_ids, "attention_mask": attention_mask}, input_num_cat],
                    outputs=[final_layer],
                    name="merged")

#define custom loss function for combining classification with regression
classification_loss = losses.SparseCategoricalCrossentropy(from_logits=False)
loss_function = "mean_absolute_error"
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
metrics = ["mae", "mse"]

final_model.compile(loss=loss_function,
                   optimizer=optimizer,
                    metrics=metrics)

In [ ]:
from tensorflow.keras.utils import plot_model
keras.utils.plot_model(final_model, show_shapes=True)

In [ ]:
final_model.fit(x=[[train_input_ids, train_attention_mask], X_train_num_cat], y=y_train,
                  validation_data=([[test_input_ids, test_attention_mask], X_test_num_cat], y_test),
                  epochs=5, batch_size = 32)

THE MODEL BELOW IS THE OLD VERSION (PROBABLY THE RIGHT ONE TOO BUT ALRIGHT)

In [ ]:
#now let's make a model for our categorical and numerical data
#define variables
optimizer = keras.optimizers.Adam(learning_rate=0.01) #tf.keras.optimizers.experimental.Adagrad(learning_rate=0.001)
loss_function = "mean_absolute_error"

#define the numeric and categorical model
input_dim_num_cat = 55 #len(X_train[0])
input_num_cat = Input(shape=input_dim_num_cat)
num_cat_layer_1 = Dense(128, activation = "relu")(input_num_cat)
num_cat_layer_2 = Dense(64, activation="relu")(num_cat_layer_1)
num_cat_layer_3 = Dense(32, activation="relu")(num_cat_layer_2)
num_cat_layer_4 = Dense(10, activation="relu")(num_cat_layer_3)

model_name = "jplu/tf-xlm-roberta-base"
num_of_classes = 10

input_ids = tf.keras.Input(shape=(max_length,), dtype='int32')
attention_mask = tf.keras.Input(shape=(max_length, ), dtype='int32')
transformer = TFAutoModelForSequenceClassification.from_pretrained(model_name, num_labels=10)
encoded = transformer({"input_ids": input_ids, "attention_mask": attention_mask})
logits = encoded[0]
probabilities = Dense(num_of_classes, activation="softmax")(logits)
#model = tf.keras.models.Model(inputs = {"input_ids": input_ids, "attention_mask": attention_mask}, outputs = probabilities)

combined_input = concatenate([probabilities, num_cat_layer_4], name="concatenated_layer")
final_layer = Dense(1, activation="linear", name="output_layer")(combined_input)
final_model = Model(inputs=[{"input_ids": input_ids, "attention_mask": attention_mask}, input_num_cat],
                    outputs=[final_layer],
                    name="merged")

#define custom loss function for combining classification with regression
classification_loss = losses.SparseCategoricalCrossentropy(from_logits=False)
loss_function = "mean_absolute_error"
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

final_model.compile(loss=loss_function,
                   optimizer=optimizer)

All model checkpoint layers were used when initializing TFXLMRobertaForSequenceClassification.

Some layers of TFXLMRobertaForSequenceClassification were not initialized from the model checkpoint at jplu/tf-xlm-roberta-base and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from tensorflow.keras.utils import plot_model
keras.utils.plot_model(final_model, show_shapes=True)

In [ ]:
final_model.fit(x=[[train_input_ids, train_attention_mask], X_train], y=y_train,
                  validation_data=([[test_input_ids, test_attention_mask], X_test], y_test),
                  epochs=5, batch_size = 32)

In [ ]:
#reshape data for randomforestregressor
y_train = np.ravel(y_train)
y_test = np.ravel(y_test)

In [ ]:
#let's fit a randomforrest on the data to see if this provides better results
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
random_forrest_regressor = RandomForestRegressor(n_estimators=100, random_state=0)
random_forrest_regressor.fit(X_train, y_train)
y_pred = random_forrest_regressor.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

In [ ]:
print("r2: {} ""\n\n"", MAE: {}".format(r2, mae))

In [ ]:
# Define the parameter grid for the grid search
param_grid = {
    'n_estimators': [50, 100, 150, 300],  # You can add more values to search over
    'max_depth': [None, 10, 20],      # You can add more values to search over
    'min_samples_split': [2, 5, 10],  # You can add more values to search over
    "criterion": [“squared_error”, “absolute_error”]
}

# Define the scoring metrics for GridSearchCV
scoring = {
    'r2': make_scorer(r2_score),
    'mae': make_scorer(mean_absolute_error),
}

In [ ]:
y_train